## Introduction

This lesson will cover:
- Connecting Claude to Robinhood's MCP server
- Querying portfolio and market data
- Building a trading agent with human-in-the-loop confirmation
- Applying safety guardrails to limit risk

## Learning Goals

After completing this lesson you will know how to:
- Use the Anthropic SDK to call Claude with MCP tools
- Design an agentic loop that handles tool calls
- Implement human confirmation before executing financial actions
- Apply practical safeguards to an AI trading agent

## Step 1 — Install dependencies

We need:
- `anthropic` — the Claude SDK
- `mcp` — the MCP client library for connecting to Robinhood's server
- `python-dotenv` — for loading environment variables

In [ ]:
%pip install -q anthropic mcp python-dotenv

## Step 2 — Imports and configuration

In [ ]:
import os
import json
import asyncio
from dotenv import load_dotenv
import anthropic
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
ROBINHOOD_MCP_URL = "https://agent.robinhood.com/mcp/trading"
MODEL = "claude-opus-4-8"  # Use the most capable model for financial reasoning

if not ANTHROPIC_API_KEY:
    raise ValueError("ANTHROPIC_API_KEY not found. Add it to your .env file.")

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print("Client ready.")

## Step 3 — Connect to the Robinhood MCP server and discover tools

When we connect to the Robinhood MCP server it returns a list of **tools** — functions
the agent can call. We convert these into the format the Anthropic SDK expects.

In [ ]:
def mcp_tool_to_anthropic(tool) -> dict:
    """Convert an MCP tool definition to the Anthropic tool schema format."""
    return {
        "name": tool.name,
        "description": tool.description or "",
        "input_schema": tool.inputSchema,
    }


async def list_robinhood_tools():
    async with streamablehttp_client(ROBINHOOD_MCP_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.list_tools()
            return result.tools


tools_raw = asyncio.run(list_robinhood_tools())
tools = [mcp_tool_to_anthropic(t) for t in tools_raw]

print(f"Found {len(tools)} tools from Robinhood MCP:\n")
for t in tools:
    print(f"  • {t['name']}: {t['description'][:80]}")

## Step 4 — Build the agentic loop

The agentic loop works like this:

1. Send user message + tools to Claude
2. If Claude returns a `tool_use` block, call the tool on the MCP server
3. Send the tool result back to Claude
4. Repeat until Claude gives a final text response

**Safety rule**: Before executing any order tool, we pause and ask the user to confirm.

In [ ]:
ORDER_TOOLS = {"place_order", "create_order", "submit_order", "buy", "sell"}


def requires_confirmation(tool_name: str, tool_input: dict) -> bool:
    """Return True if a tool call requires human approval before proceeding."""
    return any(keyword in tool_name.lower() for keyword in ["order", "buy", "sell", "trade"])


def confirm_trade(tool_name: str, tool_input: dict) -> bool:
    """Ask the user to approve or reject a trade."""
    print("\n" + "=" * 60)
    print("TRADE CONFIRMATION REQUIRED")
    print("=" * 60)
    print(f"Tool    : {tool_name}")
    print(f"Details : {json.dumps(tool_input, indent=2)}")
    print("=" * 60)
    answer = input("Approve this trade? (yes/no): ").strip().lower()
    return answer in ("yes", "y")


async def call_mcp_tool(tool_name: str, tool_input: dict) -> str:
    """Execute a tool call on the Robinhood MCP server and return the result."""
    async with streamablehttp_client(ROBINHOOD_MCP_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, tool_input)
            if result.content:
                return result.content[0].text if hasattr(result.content[0], "text") else str(result.content[0])
            return "(no result)"


async def run_trading_agent(user_message: str, max_turns: int = 10) -> str:
    """
    Run Claude in an agentic loop with Robinhood MCP tools.
    Pauses for human confirmation before any order is placed.
    """
    messages = [{"role": "user", "content": user_message}]
    system = (
        "You are a careful, research-driven trading assistant. "
        "Always check the portfolio and current prices before suggesting trades. "
        "Explain your reasoning clearly. Never place more than 1 order per request unless explicitly asked."
    )

    for turn in range(max_turns):
        response = client.messages.create(
            model=MODEL,
            max_tokens=4096,
            system=system,
            tools=tools,
            messages=messages,
        )

        # Collect text blocks for display
        for block in response.content:
            if hasattr(block, "text"):
                print(f"\nClaude: {block.text}")

        if response.stop_reason == "end_turn":
            final = next((b.text for b in response.content if hasattr(b, "text")), "")
            return final

        if response.stop_reason != "tool_use":
            break

        # Handle tool calls
        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue

            tool_name = block.name
            tool_input = block.input
            print(f"\n[Tool call] {tool_name}({json.dumps(tool_input)})")

            if requires_confirmation(tool_name, tool_input):
                approved = confirm_trade(tool_name, tool_input)
                if not approved:
                    tool_result = "Order cancelled by user."
                    print("Trade rejected.")
                else:
                    tool_result = await call_mcp_tool(tool_name, tool_input)
                    print(f"[Tool result] {tool_result[:200]}")
            else:
                tool_result = await call_mcp_tool(tool_name, tool_input)
                print(f"[Tool result] {tool_result[:200]}")

            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": tool_result,
            })

        # Append assistant turn and tool results for next loop iteration
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})

    return "Agent reached max turns without a final answer."


print("Agent loop ready.")

## Step 5 — Query your portfolio

Let's start with a read-only request. This does **not** place any trades.

In [ ]:
result = asyncio.run(run_trading_agent(
    "Show me my current portfolio. "
    "List my positions, their current values, and total buying power."
))
print("\n" + "=" * 60)
print("Final answer:")
print(result)

## Step 6 — Research a stock

Ask Claude to look up data on a specific ticker before touching any orders.

In [ ]:
TICKER = "AAPL"  # Change to any ticker you want to research

result = asyncio.run(run_trading_agent(
    f"Research {TICKER}. "
    "Get the current price, 52-week high/low, and any recent analyst notes. "
    "Give me a one-paragraph summary of whether it looks attractive right now and why."
))
print("\nFinal summary:")
print(result)

## Step 7 — Trade with human confirmation

Now we ask the agent to make a trading decision. It will:
1. Check the portfolio
2. Look up the price
3. Propose an order
4. **Pause and ask you to confirm** before executing

> You will see a `TRADE CONFIRMATION REQUIRED` prompt. Type `yes` to approve or `no` to reject.

In [ ]:
TICKER = "AAPL"  # Change to the stock you want to trade
AMOUNT = 1       # Number of shares

result = asyncio.run(run_trading_agent(
    f"I want to buy {AMOUNT} share(s) of {TICKER} if the current price is reasonable. "
    "Check the current price, compare it to the 52-week range, and place a market order if it looks good. "
    "Explain your reasoning before placing any order."
))

## Step 8 — Portfolio rebalancing (advanced)

A more sophisticated use case: ask the agent to analyze sector concentration
and suggest rebalancing trades. The agent may propose multiple orders — each
one will require your confirmation.

In [ ]:
result = asyncio.run(run_trading_agent(
    "Analyze my portfolio for sector concentration. "
    "Identify any sectors where I am overweight (>30% of portfolio) or underweight (<5%). "
    "Suggest up to 2 rebalancing trades and execute them after I approve each one individually. "
    "Do not place more than 2 orders total."
))

## Summary

In this lesson you:

- Connected Claude to Robinhood's official MCP server at `https://agent.robinhood.com/mcp/trading`
- Discovered the available trading tools automatically from the MCP server
- Built an agentic loop that lets Claude call tools iteratively to complete multi-step tasks
- Implemented **human-in-the-loop confirmation** so no trade executes without your approval
- Ran read-only queries (portfolio, stock research) and a trade with confirmation

## Key takeaways

| Concept | What we learned |
|---------|----------------|
| MCP | A standard protocol for AI agents to call external tools |
| Agentic loop | Claude reasons → calls tool → gets result → reasons again |
| Human-in-the-loop | Always pause before irreversible financial actions |
| Isolated account | Robinhood's safety design: agent account is separate from main portfolio |

## Assignment

1. Add a `max_daily_trades` guard: track how many orders have been confirmed today and refuse to place more once the limit is hit.
2. Log every tool call and result to a `trades_log.csv` file.
3. Implement a momentum strategy: ask the agent to buy the top-performing stock in your watchlist and sell the worst performer.